Tools and Technologies
used following:
Python
Hugging Face Transformers
TensorFlow or PyTorch
Jupyter Notebook / VS Code


**Task Description:**
You are required to build a token classification system using a transformer model to perform POS tagging and chunking. The task includes dataset handling, preprocessing, training, evaluation, and inference.


## 1. Install & Import Libraries

In [3]:
!pip install transformers datasets seqeval evaluate

import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification
from transformers import TrainingArguments, Trainer
import evaluate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.9 MB/s eta 0:00:00
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=d1fbdf56946c4b54991a8fde8e83d28d3ba6e9586fdf348b513240640f32d290
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


In [4]:
!pip uninstall -y datasets
!pip install datasets==2.19.0

Found existing installation: datasets 4.0.0
Uninstalling datasets-4.0.0:
  Successfully uninstalled datasets-4.0.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.0/542.0 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.0/172.0 kB 10.8 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.3.1 which is incompatible.


## 2. Load Dataset POS Tagging

In [1]:
from datasets import load_dataset

dataset = load_dataset("conll2003")
print(dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/datasets/load.py:1486: FutureWarning: The repository for conll2003 contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/conll2003
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(


Generating train split:   0%|          | 0/14041 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3250 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3453 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})


## 3. Label Mapping

In [2]:
label_list = dataset["train"].features["pos_tags"].feature.names

label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for i, label in enumerate(label_list)}

num_labels = len(label_list)

print(label_list)

['"', "''", '#', '$', '(', ')', ',', '.', ':', '``', 'CC', 'CD', 'DT', 'EX', 'FW', 'IN', 'JJ', 'JJR', 'JJS', 'LS', 'MD', 'NN', 'NNP', 'NNPS', 'NNS', 'NN|SYM', 'PDT', 'POS', 'PRP', 'PRP$', 'RB', 'RBR', 'RBS', 'RP', 'SYM', 'TO', 'UH', 'VB', 'VBD', 'VBG', 'VBN', 'VBP', 'VBZ', 'WDT', 'WP', 'WP$', 'WRB']


## 4. Load Tokenizer

In [5]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

## Tokenization + Label Alignment


In [6]:
def tokenize_and_align_labels(example):
    tokenized_inputs = tokenizer(
        example["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    labels = []
    word_ids = tokenized_inputs.word_ids()

    previous_word_idx = None

    for word_idx in word_ids:
        if word_idx is None:
            labels.append(-100)   # special tokens
        elif word_idx != previous_word_idx:
            labels.append(example["pos_tags"][word_idx])
        else:
            labels.append(-100)   # subword tokens
        previous_word_idx = word_idx

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

## 6. Apply Preprocessing

In [7]:
tokenized_dataset = dataset.map(tokenize_and_align_labels, batched=False, num_proc=1)

train_dataset = tokenized_dataset["train"]
val_dataset = tokenized_dataset["validation"]
test_dataset = tokenized_dataset["test"]

Map:   0%|          | 0/14041 [00:00<?, ? examples/s]

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

## 7. Load Model

In [8]:
model = AutoModelForTokenClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## 8. Evaluation Metric (seqeval)

In [10]:
import evaluate
metric = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = []
    true_labels = []

    for pred, lab in zip(predictions, labels):
        temp_pred = []
        temp_lab = []

        for p_, l_ in zip(pred, lab):
            if l_ != -100:
                temp_pred.append(id2label[p_])
                temp_lab.append(id2label[l_])

        true_predictions.append(temp_pred)
        true_labels.append(temp_lab)

    results = metric.compute(predictions=true_predictions, references=true_labels)

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"]
    }

## 9. Training Arguments

In [13]:
from transformers import TrainingArguments, Trainer
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_dir="./logs",
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [20]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer)

In [21]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [23]:
!pip install transformers datasets seqeval evaluate

import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification
from transformers import TrainingArguments, Trainer
import evaluate

## Train Model

In [25]:

trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.169687,0.221531,0.921250,0.921630,0.921440,0.945621
2,0.124213,0.213703,0.924976,0.925246,0.925111,0.948230


Epoch,Training Loss,Validation Loss


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: NNP seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: : seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: IN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: NN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: . seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarni

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: NNP seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: : seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: IN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: NN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: . seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarni

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3512, training_loss=0.15798602047840934, metrics={'train_runtime': 213.9592, 'train_samples_per_second': 131.249, 'train_steps_per_second': 16.414, 'total_flos': 318691831379556.0, 'train_loss': 0.15798602047840934, 'epoch': 2.0})

## 12. Evaluate Model

In [26]:
results = trainer.evaluate()
print(results)

/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: NNP seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: : seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: IN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: NN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: . seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarni

{'eval_loss': 0.21370263397693634, 'eval_precision': 0.9249763504499479, 'eval_recall': 0.9252456629867767, 'eval_f1': 0.9251109871182164, 'eval_accuracy': 0.9482302091040069, 'eval_runtime': 6.5625, 'eval_samples_per_second': 495.236, 'eval_steps_per_second': 62.019, 'epoch': 2.0}


In [27]:
model = trainer.model

In [29]:
trainer.save_model("./my_model")
tokenizer.save_pretrained("./my_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./my_model/tokenizer_config.json', './my_model/tokenizer.json')

In [31]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained("./my_model")

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

In [32]:
!ls ./results

checkpoint-1756  checkpoint-3512


In [33]:
import torch
import numpy as np

def predict(sentence, label_type="pos"):
    tokens = sentence.split()

    inputs = tokenizer(
        tokens,
        return_tensors="pt",
        is_split_into_words=True,
        truncation=True
    )

    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits
    predictions = torch.argmax(logits, dim=2).numpy()

    word_ids = inputs.word_ids()

    predicted_labels = []
    previous_word_idx = None

    for idx, word_idx in enumerate(word_ids):
        if word_idx is None:
            continue
        elif word_idx != previous_word_idx:
            predicted_labels.append(id2label[predictions[0][idx]])
        previous_word_idx = word_idx

    return list(zip(tokens, predicted_labels))

## Inference (Prediction on Custom Sentence)

In [34]:
sentence = "John works at Google in California"
output = predict(sentence)

for word, tag in output:
    print(f"{word:12} → {tag}")

John         → NNP
works        → VBZ
at           → IN
Google       → NNP
in           → IN
California   → NNP


## POS Tagging
Assigns grammatical labels to each word
Works at word level
Identifies parts of speech like noun, verb, adjective, etc.
Example tags: NNP, VBZ, IN
Focuses on what each word is
Does not depend much on neighboring words
Easier to implement and understand
Used in grammar checking and basic NLP tasks

## Chunking
Groups words into meaningful phrases
Works at phrase level
Identifies structures like noun phrases (NP), verb phrases (VP)
Example tags: B-NP, I-NP, B-VP
Focuses on how words are grouped together
Depends on context and neighboring words
More complex than POS tagging
Used in information extraction and sentence structure analysis
## One-Line Summary
POS Tagging → identifies individual word roles
Chunking → identifies groups of words (phrases)

**🔹 1. Differences between POS Tagging and Chunking**

Part-of-Speech (POS) tagging and chunking are fundamental tasks in Natural Language Processing that operate at different levels of linguistic analysis. POS tagging assigns grammatical labels such as noun, verb, or adjective to each individual word in a sentence, making it a word-level task. In contrast, chunking groups words into meaningful phrases such as noun phrases (NP) and verb phrases (VP), making it a phrase-level task. While POS tagging focuses on identifying the role of each word, chunking focuses on how words are structured together in a sentence. Chunking is generally more complex because it depends on the context and relationships between neighboring words, whereas POS tagging can often be performed independently for each word. Thus, chunking builds upon POS tagging and provides a higher-level understanding of sentence structure.

**🔹 2. Challenges Faced**

During the implementation of the token classification model using BERT, several challenges were encountered. One major challenge was handling subword tokenization, where a single word is split into multiple tokens by the tokenizer, making label alignment difficult. Assigning correct labels while ignoring special tokens using the value -100 was another critical step that required careful handling. Additionally, compatibility issues with library versions (such as datasets and transformers) caused runtime errors that needed debugging. Training the model also required significant computational time, especially when using larger models like BERT. Managing these challenges was essential to ensure correct model performance.

**🔹 3. Observations and Insights**

Through this assignment, it was observed that transformer models like BERT are highly effective for sequence labeling tasks such as POS tagging and chunking because they capture contextual relationships between words. POS tagging was found to be relatively easier and faster to train compared to chunking, as chunking requires understanding of phrase-level dependencies. Proper preprocessing, especially tokenization and label alignment, plays a crucial role in achieving good performance. It was also observed that even small mistakes in preprocessing can significantly affect model accuracy. Overall, the assignment provided valuable insights into how modern NLP models handle language understanding tasks efficiently.